## 1. Hugging Face Login

In [ ]:
%pip install -q huggingface_hub ijson

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [ ]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class TransformerConfig:
    vocab_size: int
    pad_id: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    n_layers: int = 6
    dropout: float = 0.1
    max_src_len: int = 2048
    max_tgt_len: int = 2048
    pos_emb_type: str = "rope"
    rope_base: float = 10000.0


class EmbeddingDropout(nn.Module):
    # Preserve the embedding dropout and configured length limit without adding PE.
    def __init__(self, max_len, dropout):
        super().__init__()
        self.max_len = max_len
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        if x.size(1) > self.max_len:
            raise ValueError('Sequence exceeds configured context length')
        return self.dropout(x)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        if head_dim % 2:
            raise ValueError('RoPE requires an even head dimension')
        inv_freq = base ** (
            -torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        )
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, x):
        # x: [batch, heads, length, head_dim]. Split-half pairs: (i, i + d/2).
        # Generation recomputes the full decoder sequence, so positions start at 0.
        with torch.autocast(device_type=x.device.type, enabled=False):
            positions = torch.arange(x.size(2), device=x.device, dtype=torch.float32)
            angles = torch.outer(positions, self.inv_freq.float())
            cos = angles.cos()[None, None, :, :]
            sin = angles.sin()[None, None, :, :]
        cos, sin = cos.to(x.dtype), sin.to(x.dtype)
        first, second = x.chunk(2, dim=-1)
        return torch.cat((first * cos - second * sin,
                          first * sin + second * cos), dim=-1)


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, use_rope=False, rope_base=10000.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.rope = RotaryEmbedding(self.d_head, rope_base) if use_rope else None
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        # [batch, length, d_model] -> [batch, heads, length, d_head]
        b, length, _ = x.shape
        return x.reshape(b, length, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, query, key, value, allowed, return_attention=False):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        if self.rope is not None:
            q = self.rope(q)
            k = self.rope(k)
        # True means this key may be attended to, including the causal mask.
        weights = None
        if return_attention:
            # Keep explicit probabilities for attention visualizations.
            with torch.autocast(device_type=q.device.type, enabled=False):
                scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(self.d_head)
                scores = scores.masked_fill(~allowed, float('-inf'))
                weights = scores.softmax(dim=-1)
            context = self.dropout(weights.to(v.dtype)) @ v
        else:
            # Let PyTorch select a supported backend; dropout must be 0 at eval.
            context = F.scaled_dot_product_attention(
                q, k, v, attn_mask=allowed,
                dropout_p=self.dropout.p if self.training else 0.0,
                is_causal=False,  # Causality is already present in allowed.
            )
        b, _, length, _ = context.shape
        context = context.transpose(1, 2).contiguous().reshape(b, length, -1)
        output = self.out_proj(context)
        return output, weights.mean(dim=1) if return_attention else None

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, source_mask):
        attended, _ = self.attention(x, x, x, source_mask)
        x = self.norm1(x + self.dropout(attended))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.cross_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=False)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.norm3 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, memory, target_mask, source_mask, return_attention=False):
        attended, _ = self.self_attention(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attended))
        attended, weights = self.cross_attention(x, memory, memory, source_mask,
                                                  return_attention)
        x = self.norm2(x + self.dropout(attended))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, weights

In [ ]:
class LegalTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        if cfg.pos_emb_type != 'rope':
            raise ValueError('This architecture expects a RoPE configuration')
        self.cfg = cfg
        # Separate weights make the three roles easy to inspect.
        self.source_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.target_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.source_position = EmbeddingDropout(cfg.max_src_len, cfg.dropout)
        self.target_position = EmbeddingDropout(cfg.max_tgt_len, cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.output = nn.Linear(cfg.d_model, cfg.vocab_size)
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)
        with torch.no_grad():
            self.source_embedding.weight[cfg.pad_id].zero_()
            self.target_embedding.weight[cfg.pad_id].zero_()

    def encode(self, source_ids):
        source_mask = source_ids.ne(self.cfg.pad_id)[:, None, None, :]
        x = self.source_embedding(source_ids) * math.sqrt(self.cfg.d_model)
        x = self.source_position(x)
        for layer in self.encoder:
            x = layer(x, source_mask)
        return x, source_mask

    def decode(self, target_ids, memory, source_mask, return_attention=False):
        length = target_ids.size(1)
        causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()
        valid_keys = target_ids.ne(self.cfg.pad_id)[:, None, None, :]
        target_mask = valid_keys & causal[None, None, :, :]
        x = self.target_embedding(target_ids) * math.sqrt(self.cfg.d_model)
        x = self.target_position(x)
        weights = None
        for index, layer in enumerate(self.decoder):
            x, weights = layer(x, memory, target_mask, source_mask,
                               return_attention and index == len(self.decoder) - 1)
        logits = self.output(x)
        return (logits, weights) if return_attention else logits

    def forward(self, source_ids, target_ids, return_attention=False):
        memory, source_mask = self.encode(source_ids)
        return self.decode(target_ids, memory, source_mask, return_attention)

# **Phase 2**

In [ ]:
import os
import json
import math
import time
import random
import hashlib
import zipfile
from pathlib import Path
from dataclasses import asdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from huggingface_hub import hf_hub_download

# Uses the best final weights trained above. For Phase-2-only runs, edit this path.
PHASE1_DIR = Path('/kaggle/working/cs781_phase1_encdec_rope_2048_400m/legal-transformer-final')
CHECKPOINT_PATH = PHASE1_DIR / 'model.pt'
TOKENIZER_PATH = PHASE1_DIR / 'tokenizer.json'

ADAPTER_TYPE = 'lorma_plus'  # 'lora' or 'lorma_plus'
RANK = 8
ALPHA = 16
ATTENTION_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'out_proj']
FFN_TARGETS = ['linear1', 'linear2']  # [] for attention-only

SEED = 42
MAX_LENGTH = 1024
LEFT_CONTEXT = 256
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 3e-4
SCHEDULER_PATIENCE = 3
EARLY_STOPPING_PATIENCE = 7
B_WEIGHT_MULTIPLIER = 1.5
O_WEIGHT = 0.3
RUN_NAME = f'encdec_rope_2048_400m_{ADAPTER_TYPE}_qkvo_ffn_run1'
PHASE2_OUTPUT_DIR = Path('/kaggle/working') / RUN_NAME
PHASE2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Uses HF_TOKEN from the environment or an existing Hugging Face login.
# On Kaggle, set your secret once if authentication is needed:
# from kaggle_secrets import UserSecretsClient
# os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
HF_TOKEN = os.environ.get('HF_TOKEN')


In [ ]:
with open(
    TOKENIZER_PATH,
    "r",
    encoding="utf-8",
) as f:
    state = json.load(f)

tokenizer = BytelevelBPE(vocab_size=state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(state["vocab"])

tokenizer.stoi = {
    symbol: i for i, symbol in enumerate(state["vocab"])
}
tokenizer.itos = {
    i: symbol for symbol, i in tokenizer.stoi.items()
}

tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]
tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}
tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}


print("Tokenizer loaded:", tokenizer.get_vocab_size())


In [ ]:
import json, hashlib
from huggingface_hub import hf_hub_download
from pathlib import Path

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

files = [
    "Train/fold1-text.json",
    "Train/fold1-gold.json",
    "Train/fold2-text.json",
    "Train/fold2-gold.json",
    "Train/labels.txt",
    "Test/fold3-text.json",
]

paths = {}

for filename in files:
    paths[filename] = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=f"Phase_2_LNER/{filename}",
        repo_type="dataset",
        token=HF_TOKEN,
    )

fold1_texts = load_json(paths["Train/fold1-text.json"])
fold1_gold = load_json(paths["Train/fold1-gold.json"])

fold2_texts = load_json(paths["Train/fold2-text.json"])
fold2_gold = load_json(paths["Train/fold2-gold.json"])

# Prevent duplicate IDs from silently overwriting documents.
if set(fold1_texts) & set(fold2_texts):
    raise ValueError("The two folds contain duplicate document IDs.")

all_texts = {**fold1_texts, **fold2_texts}
all_gold = {**fold1_gold, **fold2_gold}

if set(all_texts) != set(all_gold):
    raise ValueError("Document IDs and annotation IDs do not match.")

# Fixed seed keeps the split identical across experiments.
doc_ids = sorted(all_texts)
random.Random(SEED).shuffle(doc_ids)

split_index = int(0.9 * len(doc_ids))

train_ids = doc_ids[:split_index]
val_ids = doc_ids[split_index:]

train_texts = {doc_id: all_texts[doc_id] for doc_id in train_ids}
train_gold = {doc_id: all_gold[doc_id] for doc_id in train_ids}

val_texts = {doc_id: all_texts[doc_id] for doc_id in val_ids}
val_gold = {doc_id: all_gold[doc_id] for doc_id in val_ids}

print("Training documents:", len(train_texts))
print("Validation documents:", len(val_texts))

test_texts = load_json(paths["Test/fold3-text.json"])


In [ ]:
with open(paths["Train/labels.txt"], encoding="utf-8") as f:
    entity_types = [line.strip() for line in f if line.strip()]

tags = ["O"]

for entity in entity_types:
    tags.append(f"B-{entity}")
    tags.append(f"I-{entity}")

label2id = {tag: i for i, tag in enumerate(tags)}
id2label = {i: tag for tag, i in label2id.items()}

print("Number of labels:", len(tags))  # 25
print(label2id)


In [ ]:
split_token_ids = set()

import re
import string


def encode_with_offsets(text, tokenizer):
    input_ids = []
    offsets = []

    punctuation = set(string.punctuation.encode("ascii"))

    # Process both whitespace and non-whitespace segments.
    for match in re.finditer(r"\s+|\S+", text):
        segment = match.group()
        segment_start = match.start()

        # Preserve whitespace and its exact character positions.
        if segment.isspace():
            for char_index, char in enumerate(segment):
                start = segment_start + char_index

                for byte in char.encode("utf-8"):
                    input_ids.append(tokenizer.stoi[str(byte)])
                    offsets.append((start, start + 1))

            continue

        # Map each UTF-8 byte to its original character position.
        byte_to_char = []

        for char_index, char in enumerate(segment):
            byte_to_char.extend(
                [segment_start + char_index]
                * len(char.encode("utf-8"))
            )

        byte_position = 0

        for symbol in tokenizer.apply_bpe(segment):
            token_id = tokenizer.stoi[symbol]
            token_bytes = tokenizer.symbol_to_bytes[symbol]

            split_punctuation = (
                len(token_bytes) > 1
                and any(byte in punctuation for byte in token_bytes)
            )

            split_boundary = token_id in split_token_ids

            if token_bytes and (
                split_punctuation or split_boundary
            ):
                pieces = [str(byte) for byte in token_bytes]

                if tokenizer._symbol_ends_word(symbol):
                    pieces.append("</w>")
            else:
                pieces = [symbol]

            for piece in pieces:
                byte_length = len(
                    tokenizer.symbol_to_bytes[piece]
                )

                input_ids.append(tokenizer.stoi[piece])

                if byte_length == 0:
                    offsets.append(None)
                else:
                    start = byte_to_char[byte_position]
                    end = (
                        byte_to_char[
                            byte_position + byte_length - 1
                        ] + 1
                    )

                    offsets.append((start, end))

                byte_position += byte_length

        if byte_position != len(byte_to_char):
            raise ValueError(
                f"Incomplete byte coverage for {segment!r}"
            )

    return input_ids, offsets


In [ ]:
# Identify training boundaries before assigning BIO labels.
for doc_id, text in train_texts.items():
    ids, offsets = encode_with_offsets(text, tokenizer)
    boundaries = sorted({
        boundary
        for annotation in train_gold[doc_id]
        for boundary in annotation[:2]
    })
    for token_id, offset in zip(ids, offsets):
        if offset is not None:
            start, end = offset
            if any(start < boundary < end for boundary in boundaries):
                split_token_ids.add(token_id)

print('Training-derived split IDs:', sorted(split_token_ids))


In [ ]:
def make_ner_labels(offsets, annotations, label2id):
    labels = [
        -100 if offset is None else label2id["O"]
        for offset in offsets
    ]

    boundary_cases = []

    for start, end, entity_type, *_ in sorted(annotations):
        indices = [
            i
            for i, offset in enumerate(offsets)
            if offset is not None
            and offset[0] < end
            and offset[1] > start
        ]

        if not indices:
            raise ValueError(
                f"No tokens overlap {(start, end, entity_type)}"
            )

        if any(labels[i] != label2id["O"] for i in indices):
            raise ValueError(
                f"Two entities share tokens near "
                f"{(start, end, entity_type)}"
            )

        first, last = indices[0], indices[-1]
        token_span = (offsets[first][0], offsets[last][1])

        if token_span != (start, end):
            boundary_cases.append({
                "gold_span": (start, end),
                "token_span": token_span,
                "label": entity_type,
            })

        labels[first] = label2id[f"B-{entity_type}"]

        for i in indices[1:]:
            labels[i] = label2id[f"I-{entity_type}"]

    return labels, boundary_cases


In [ ]:
split_token_ids = {
    558, 884, 1048, 1229,
    1244, 1709, 3351, 9880,
}

In [ ]:
def encode_documents(texts, annotations):
    encoded = {}

    for doc_id, text in texts.items():
        input_ids, offsets = encode_with_offsets(text, tokenizer)

        labels, boundary_cases = make_ner_labels(
            offsets,
            annotations[doc_id],
            label2id,
        )

        encoded[doc_id] = {
            "input_ids": input_ids,
            "labels": labels,
            "offsets": offsets,
            "boundary_cases": boundary_cases,
        }

    return encoded


encoded_train = encode_documents(train_texts, train_gold)
encoded_val = encode_documents(val_texts, val_gold)


In [ ]:
import pandas as pd


def analyze_gold_token_alignment(texts, gold, encoded_documents):
    records = []

    for doc_id, text in texts.items():
        offsets = encoded_documents[doc_id]["offsets"]

        for entity_idx, annotation in enumerate(gold[doc_id]):
            gold_start, gold_end, label, *_ = annotation

            overlapping = [
                (i, offset[0], offset[1])
                for i, offset in enumerate(offsets)
                if offset is not None
                and offset[0] < gold_end
                and offset[1] > gold_start
            ]

            row = {
                "doc_id": doc_id,
                "entity_idx": entity_idx,
                "label": label,
                "gold_text": text[gold_start:gold_end],
                "gold_start": gold_start,
                "gold_end": gold_end,
                "first_token_idx": None,
                "last_token_idx": None,
                "token_start": None,
                "token_end": None,
                "start_diff": None,
                "end_diff": None,
                "prefix_extra": "",
                "suffix_extra": "",
                "token_text": "",
                "alignment_type": "NO_TOKEN",
            }

            if overlapping:
                first_idx = overlapping[0][0]
                last_idx = overlapping[-1][0]

                token_start = min(item[1] for item in overlapping)
                token_end = max(item[2] for item in overlapping)

                if token_start == gold_start and token_end == gold_end:
                    alignment_type = "EXACT"

                elif token_start == gold_start and token_end > gold_end:
                    alignment_type = "SUFFIX_ONLY"

                elif token_start < gold_start and token_end == gold_end:
                    alignment_type = "PREFIX_ONLY"

                elif token_start < gold_start and token_end > gold_end:
                    alignment_type = "PREFIX_AND_SUFFIX"

                elif token_start > gold_start and token_end < gold_end:
                    alignment_type = "INSIDE_GOLD"

                elif token_start > gold_start:
                    alignment_type = "START_MISMATCH"

                elif token_end < gold_end:
                    alignment_type = "END_MISMATCH"

                else:
                    alignment_type = "OTHER"

                row.update({
                    "first_token_idx": first_idx,
                    "last_token_idx": last_idx,
                    "token_start": token_start,
                    "token_end": token_end,
                    "start_diff": token_start - gold_start,
                    "end_diff": token_end - gold_end,
                    "prefix_extra": (
                        text[token_start:gold_start]
                        if token_start < gold_start else ""
                    ),
                    "suffix_extra": (
                        text[gold_end:token_end]
                        if token_end > gold_end else ""
                    ),
                    "token_text": text[token_start:token_end],
                    "alignment_type": alignment_type,
                })

            records.append(row)

    if not records:
        raise ValueError("No gold entities found.")

    all_df = pd.DataFrame(records)

    alignment_counts = (
        all_df["alignment_type"].value_counts().to_dict()
    )

    total = len(all_df)
    exact = alignment_counts.get("EXACT", 0)

    summary = {
        "total_gold_entities": total,
        "exact": exact,
        "mismatched": total - exact,
        "exact_percentage": 100 * exact / total,
        "mismatch_percentage": 100 * (total - exact) / total,
        "alignment_counts": alignment_counts,
    }

    entity_stats = (
        all_df.groupby("label")
        .agg(
            total=("alignment_type", "size"),
            exact=("alignment_type", lambda x: (x == "EXACT").sum()),
            suffix_only=(
                "alignment_type",
                lambda x: (x == "SUFFIX_ONLY").sum(),
            ),
            prefix_only=(
                "alignment_type",
                lambda x: (x == "PREFIX_ONLY").sum(),
            ),
            prefix_suffix=(
                "alignment_type",
                lambda x: (x == "PREFIX_AND_SUFFIX").sum(),
            ),
            mismatched=(
                "alignment_type",
                lambda x: (x != "EXACT").sum(),
            ),
        )
        .reset_index()
    )

    entity_stats["exact_pct"] = (
        100 * entity_stats["exact"] / entity_stats["total"]
    )

    entity_stats["mismatch_pct"] = (
        100 * entity_stats["mismatched"] / entity_stats["total"]
    )

    entity_stats = entity_stats.sort_values(
        "mismatch_pct", ascending=False
    )

    mismatches_df = all_df[
        all_df["alignment_type"] != "EXACT"
    ].copy()

    return summary, entity_stats, mismatches_df, all_df


In [ ]:
summary, entity_stats, mismatches_df, all_df = (
    analyze_gold_token_alignment(
        texts=train_texts,
        gold=train_gold,
        encoded_documents=encoded_train,
    )
)

print(summary)
display(entity_stats)

display(
    mismatches_df[
        [
            "doc_id",
            "label",
            "gold_text",
            "token_text",
            "prefix_extra",
            "suffix_extra",
            "alignment_type",
        ]
    ].head(30)
)


In [ ]:
val_summary, val_entity_stats, val_mismatches_df, val_all_df = (
    analyze_gold_token_alignment(
        texts=val_texts,
        gold=val_gold,
        encoded_documents=encoded_val,
    )
)

print(val_summary)
display(val_entity_stats)
display(val_mismatches_df.head(30))


In [ ]:
def create_windows(documents, max_length=MAX_LENGTH,
                   left_context=LEFT_CONTEXT):
    if not 0 <= left_context < max_length:
        raise ValueError("Require 0 <= left_context < max_length.")

    windows = []

    for doc_id, record in documents.items():
        total_tokens = len(record["input_ids"])
        prediction_start = 0

        while prediction_start < total_tokens:
            start = max(0, prediction_start - left_context)
            end = min(start + max_length, total_tokens)

            context_length = prediction_start - start

            labels = list(record["labels"][start:end])
            labels[:context_length] = [-100] * context_length

            windows.append({
                "doc_id": doc_id,
                "token_start": start,
                "context_length": context_length,
                "input_ids": record["input_ids"][start:end],
                "labels": labels,
                "offsets": record["offsets"][start:end],
            })

            prediction_start = end

    return windows


train_windows = create_windows(encoded_train)
val_windows = create_windows(encoded_val)


In [ ]:
NUM_LABELS = len(label2id)
counts = torch.zeros(NUM_LABELS)

for example in train_windows:
    for label_id in example["labels"]:
        if label_id != -100:
            counts[label_id] += 1

total = counts.sum()

weights = torch.ones(NUM_LABELS)
entity_ids = []

for entity_type in entity_types:
    b_id = label2id[f"B-{entity_type}"]
    i_id = label2id[f"I-{entity_type}"]

    count = counts[b_id] + counts[i_id]
    class_weight = (total / count.clamp(min=1)) ** 0.25

    weights[b_id] = class_weight * B_WEIGHT_MULTIPLIER
    weights[i_id] = class_weight

    entity_ids.extend([b_id, i_id])

weights[entity_ids] /= weights[entity_ids].mean()
weights[label2id["O"]] = O_WEIGHT

for i in range(NUM_LABELS):
    print(f"{id2label[i]:15s} {weights[i].item():.4f}")


In [ ]:
for name, documents, windows in [
    ("Train", encoded_train, train_windows),
    ("Validation", encoded_val, val_windows),
]:
    boundary_count = sum(
        len(record["boundary_cases"])
        for record in documents.values()
    )

    print(
        f"{name}: {len(documents)} documents, "
        f"{len(windows)} windows, "
        f"{boundary_count} approximate entity boundaries"
    )


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

pad_id = tokenizer.stoi["<pad>"]

def collate_batch(examples):
    input_ids = [
        torch.tensor(example["input_ids"], dtype=torch.long)
        for example in examples
    ]

    labels = [
        torch.tensor(example["labels"], dtype=torch.long)
        for example in examples
    ]

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=pad_id,
        ),
        "labels": pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        ),
        "doc_ids": [example["doc_id"] for example in examples],
        "token_starts": [
            example["token_start"] for example in examples
        ],
        "context_lengths": [
            example["context_length"] for example in examples
        ],
        "offsets": [example["offsets"] for example in examples],
    }


In [ ]:
train_loader = DataLoader(
    train_windows,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_windows,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)


In [ ]:
batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Label shape:", batch["labels"].shape)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


In [ ]:
class EncoderDecoderNER(nn.Module):
    def __init__(self, pretrained_model, num_labels):
        super().__init__()
        self.cfg = pretrained_model.cfg
        self.source_embedding = pretrained_model.source_embedding
        self.target_embedding = pretrained_model.target_embedding
        self.source_position = pretrained_model.source_position
        self.target_position = pretrained_model.target_position
        self.encoder = pretrained_model.encoder
        self.decoder = pretrained_model.decoder
        self.classifier = nn.Linear(self.cfg.d_model, num_labels)

    def forward(self, input_ids):
        length = input_ids.size(1)
        if length > min(self.cfg.max_src_len, self.cfg.max_tgt_len):
            raise ValueError('Window exceeds the Phase 1 context limit.')
        source_mask = input_ids.ne(self.cfg.pad_id)[:, None, None, :]
        memory = self.source_position(
            self.source_embedding(input_ids) * math.sqrt(self.cfg.d_model)
        )
        for layer in self.encoder:
            memory = layer(memory, source_mask)

        causal = torch.ones(
            length, length, dtype=torch.bool, device=input_ids.device
        ).tril()
        target_mask = source_mask & causal[None, None, :, :]
        x = self.target_position(
            self.target_embedding(input_ids) * math.sqrt(self.cfg.d_model)
        )
        for layer in self.decoder:
            x, _ = layer(x, memory, target_mask, source_mask)
        return self.classifier(x)


In [ ]:
class LoRA(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)

        self.scale = alpha / rank

        self.A = nn.Linear(
            base_layer.in_features,
            rank,
            bias=False,
        )

        self.B = nn.Linear(
            rank,
            base_layer.out_features,
            bias=False,
        )

        self.A.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )
        self.B.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = self.base(x)
        correction = self.B(self.A(x))

        return original + self.scale * correction


In [ ]:
class LoRMAPlus(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)

        out_dim = base_layer.out_features
        self.scale = alpha / rank

        self.A = nn.Linear(out_dim, rank, bias=False)
        self.B = nn.Linear(rank, out_dim, bias=False)

        self.A.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )
        self.B.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = self.base(x)
        correction = self.B(self.A(original))

        return original + self.scale * correction


In [ ]:
def insert_adapters(model, adapter_type, rank, alpha,
                    attention_targets, ffn_targets):
    if rank < 1:
        raise ValueError('Rank must be positive.')
    Adapter = {'lora': LoRA, 'lorma_plus': LoRMAPlus}[adapter_type]
    model.requires_grad_(False)

    attention_modules = [layer.attention for layer in model.encoder]
    for layer in model.decoder:
        attention_modules.extend([layer.self_attention, layer.cross_attention])

    for attention in attention_modules:
        for name in attention_targets:
            base = getattr(attention, name)
            if not isinstance(base, nn.Linear):
                raise ValueError('Create a fresh model before inserting adapters.')
            setattr(attention, name, Adapter(base, rank, alpha))

    for layer in list(model.encoder) + list(model.decoder):
        for name in ffn_targets:
            base = getattr(layer.ffn, name)
            if not isinstance(base, nn.Linear):
                raise ValueError('Create a fresh model before inserting adapters.')
            setattr(layer.ffn, name, Adapter(base, rank, alpha))

    model.classifier.requires_grad_(True)


def build_ner_model():
    # Called for every new experiment; never reuses generation_model.
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
    cfg = TransformerConfig(**checkpoint['model_config'])
    if MAX_LENGTH > min(cfg.max_src_len, cfg.max_tgt_len):
        raise ValueError('Reduce MAX_LENGTH to the Phase 1 context limit.')
    if cfg.vocab_size != tokenizer.get_vocab_size() or cfg.pad_id != pad_id:
        raise ValueError('Checkpoint and tokenizer configuration do not match.')
    phase1_model = LegalTransformer(cfg)
    phase1_model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    del checkpoint
    torch.manual_seed(SEED)
    ner = EncoderDecoderNER(phase1_model, len(label2id))
    del phase1_model
    insert_adapters(ner, ADAPTER_TYPE, RANK, ALPHA,
                    ATTENTION_TARGETS, FFN_TARGETS)
    return ner.to(device)


if 'ner_model' in globals():
    del ner_model
if device.type == 'cuda':
    torch.cuda.empty_cache()
ner_model = build_ner_model()

adaptation_config = dict(
    adapter_type=ADAPTER_TYPE, rank=RANK, alpha=ALPHA,
    attention_targets=ATTENTION_TARGETS.copy(),
    ffn_targets=FFN_TARGETS.copy(),
    attention_locations=['encoder.attention', 'decoder.self_attention',
                         'decoder.cross_attention'],
    encoder_causal=False, decoder_causal=True,
    decoder_input='same_text_tokens_without_shift',
)
total = sum(p.numel() for p in ner_model.parameters())
trainable = sum(p.numel() for p in ner_model.parameters() if p.requires_grad)
print('Adaptation:', adaptation_config)
print(f'Total: {total:,} | Trainable: {trainable:,} ({100*trainable/total:.2f}%)')


In [ ]:
def decode_entities(predicted_ids, offsets):
    entities = []
    current = None

    for label_id, offset in zip(predicted_ids, offsets):
        if offset is None:
            continue

        tag = id2label[label_id]
        start, end = offset

        if tag == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue

        prefix, entity_type = tag.split("-", 1)

        if (
            prefix == "I"
            and current is not None
            and current[2] == entity_type
        ):
            current[1] = end
        else:
            if current is not None:
                entities.append(current)

            current = [start, end, entity_type]

    if current is not None:
        entities.append(current)

    return entities


In [ ]:
import numpy as np


def viterbi_bio_decode(logits, id2label):
    # logits: [number_of_tokens, number_of_labels]
    length, num_labels = logits.shape

    if length == 0:
        return []

    # Rows = previous tag, columns = current tag.
    transitions = np.zeros(
        (num_labels, num_labels),
        dtype=np.float32,
    )

    start_scores = np.zeros(num_labels, dtype=np.float32)

    for current_id in range(num_labels):
        tag = id2label[current_id]

        if tag.startswith("I-"):
            entity_type = tag[2:]

            # An I-tag cannot start a document.
            start_scores[current_id] = -np.inf

            # An I-X tag must follow B-X or I-X.
            for previous_id in range(num_labels):
                previous_tag = id2label[previous_id]

                if previous_tag not in (
                    f"B-{entity_type}",
                    f"I-{entity_type}",
                ):
                    transitions[previous_id, current_id] = -np.inf

    scores = logits[0] + start_scores

    backpointers = np.zeros(
        (length, num_labels),
        dtype=np.int32,
    )

    for position in range(1, length):
        candidates = scores[:, None] + transitions

        best_previous = candidates.argmax(axis=0)
        backpointers[position] = best_previous

        scores = (
            candidates[best_previous, np.arange(num_labels)]
            + logits[position]
        )

    # Recover the best complete sequence.
    best_last = int(scores.argmax())
    predictions = [best_last]

    for position in range(length - 1, 0, -1):
        best_last = int(backpointers[position, best_last])
        predictions.append(best_last)

    predictions.reverse()
    return predictions

In [ ]:
@torch.no_grad()
def predict_entities(
    model,
    loader,
    encoded_documents,
    use_viterbi=True,
):
    model.eval()
    device = next(model.parameters()).device
    num_labels = len(id2label)

    document_logits = {
        doc_id: np.zeros(
            (len(record["input_ids"]), num_labels),
            dtype=np.float32,
        )
        for doc_id, record in encoded_documents.items()
    }

    covered = {
        doc_id: np.zeros(
            len(record["input_ids"]),
            dtype=bool,
        )
        for doc_id, record in encoded_documents.items()
    }

    for batch in loader:
        inputs = batch["input_ids"].to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model(inputs)

        logits = logits.float().cpu().numpy()

        for i, doc_id in enumerate(batch["doc_ids"]):
            chunk_start = batch["token_starts"][i]
            context_length = batch["context_lengths"][i]
            length = len(batch["offsets"][i])

            start = chunk_start + context_length
            end = chunk_start + length

            document_logits[doc_id][start:end] = (
                logits[i, context_length:length]
            )

            covered[doc_id][start:end] = True

    results = {}

    for doc_id, record in encoded_documents.items():
        if not covered[doc_id].all():
            raise ValueError(
                f"Missing predictions for document {doc_id}"
            )

        # Skip ignored word markers, keeping text-bearing tokens.
        valid_positions = [
            i
            for i, offset in enumerate(record["offsets"])
            if offset is not None
        ]

        valid_logits = document_logits[doc_id][valid_positions]

        valid_offsets = [
            record["offsets"][i]
            for i in valid_positions
        ]

        if use_viterbi:
            predicted_ids = viterbi_bio_decode(
                valid_logits,
                id2label,
            )
        else:
            predicted_ids = valid_logits.argmax(axis=-1).tolist()

        results[doc_id] = decode_entities(
            predicted_ids,
            valid_offsets,
        )

    return results

In [ ]:
def strict_ner_scores(predictions, gold):
    if set(predictions) != set(gold):
        raise ValueError("Prediction and gold document IDs do not match.")

    per_label = {}

    for entity_type in entity_types:
        predicted = {
            (doc_id, start, end)
            for doc_id, entities in predictions.items()
            for start, end, label in entities
            if label == entity_type
        }

        actual = {
            (doc_id, annotation[0], annotation[1])
            for doc_id, entities in gold.items()
            for annotation in entities
            if annotation[2] == entity_type
        }

        correct = len(predicted & actual)

        precision = correct / len(predicted) if predicted else 0.0
        recall = correct / len(actual) if actual else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[entity_type] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": len(actual),
        }

    scores = {
        metric: sum(row[metric] for row in per_label.values())
        / len(entity_types)
        for metric in ("precision", "recall", "f1")
    }

    return scores, per_label


In [ ]:
import os
import torch
import torch.nn.functional as F
from dataclasses import asdict


@torch.no_grad()
def evaluate_ner_loss(model, loader):
    model.eval()

    device = next(model.parameters()).device
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        inputs = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model(inputs)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum",
            )

        total_loss += loss.item()
        total_tokens += labels.ne(-100).sum().item()

    if total_tokens == 0:
        raise ValueError("No labelled tokens in the validation loader.")

    return total_loss / total_tokens


In [ ]:
gold_predictions = {
    d: decode_entities([x if x != -100 else label2id['O'] for x in r['labels']], r['offsets'])
    for d,r in encoded_val.items()
}
print('Gold-label round-trip:', strict_ner_scores(gold_predictions, val_gold)[0])
run_config = dict(method=ADAPTER_TYPE, seed=SEED, batch_size=BATCH_SIZE,
    epochs=EPOCHS, lr=LEARNING_RATE, max_length=MAX_LENGTH, left_context=LEFT_CONTEXT,
    encoder_causal=False, decoder_causal=True, preserve_whitespace=True, split_punctuation=True,
    split_token_ids=sorted(split_token_ids), train_ids=train_ids, val_ids=val_ids,
    class_weights=weights.tolist(), checkpoint_path=str(CHECKPOINT_PATH),
    tokenizer_sha256=hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest())
run_config['adaptation_config'] = adaptation_config
(PHASE2_OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2))


In [ ]:
def train_ner(model, train_loader, val_loader, epochs, lr, output_dir, patience=10):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=SCHEDULER_PATIENCE)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    class_weights = weights.to(device)
    history = []
    best_f1, bad_epochs = -1.0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = weight_sum = 0.0
        started = time.perf_counter()
        for step, batch in enumerate(train_loader, 1):
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            valid = labels != -100
            if not valid.any():
                continue
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = model(inputs)
                loss = F.cross_entropy(logits.reshape(-1,len(label2id)), labels.reshape(-1),
                                       weight=class_weights, ignore_index=-100)
            if not torch.isfinite(loss):
                raise ValueError('Non-finite training loss.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            mass = class_weights[labels[valid]].sum().item()
            loss_sum += loss.item()*mass
            weight_sum += mass
            if step % 50 == 0:
                print(f'Epoch {epoch} | Batch {step}/{len(train_loader)} | Loss {loss.item():.4f}', flush=True)
            del logits, loss, inputs, labels, valid
        optimizer.zero_grad(set_to_none=True)
        if weight_sum == 0:
            raise ValueError('No labelled training tokens.')
        val_loss = evaluate_ner_loss(model, val_loader)
        predictions = predict_entities(model, val_loader, encoded_val)
        scores, per_label = strict_ner_scores(predictions, val_gold)
        scheduler.step(scores['f1'])
        row = dict(epoch=epoch, weighted_train_loss=loss_sum/weight_sum,
                   unweighted_val_loss=val_loss, **scores,
                   next_lr=optimizer.param_groups[0]['lr'], seconds=time.perf_counter()-started)
        history.append(row)
        print(row, flush=True)
        (output_dir / 'history.json').write_text(json.dumps(history, indent=2))
        if scores['f1'] > best_f1:
            best_f1, bad_epochs = scores['f1'], 0
            saved = dict(model_state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
                model_config=asdict(model.cfg), label2id=label2id, run_config=run_config,
                adaptation_config=adaptation_config,
                epoch=epoch, validation_scores=scores, per_label=per_label)
            temporary = output_dir / 'best_f1.tmp'
            torch.save(saved, temporary)
            os.replace(temporary, output_dir / 'best_f1.pt')
            del saved
            (output_dir / 'validation_predictions.json').write_text(json.dumps(predictions, ensure_ascii=False))
        else:
            bad_epochs += 1
        if bad_epochs >= patience:
            print('Early stopping.')
            break
    return history, epoch


In [ ]:
import shutil
shutil.copy2(TOKENIZER_PATH, PHASE2_OUTPUT_DIR / 'tokenizer.json')
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

history, last_epoch = train_ner(
    model=ner_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=PHASE2_OUTPUT_DIR,
    patience=EARLY_STOPPING_PATIENCE,
)


In [ ]:
checkpoint = torch.load(
    PHASE2_OUTPUT_DIR / 'best_f1.pt', map_location='cpu', weights_only=True
)
if checkpoint['label2id'] != label2id:
    raise ValueError('Checkpoint label mapping differs from this run.')
if checkpoint['adaptation_config'] != adaptation_config:
    raise ValueError('Recreate the same adapter architecture before loading.')
if checkpoint['run_config']['tokenizer_sha256'] != hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest():
    raise ValueError('Use the tokenizer belonging to this checkpoint.')
ner_model.load_state_dict(checkpoint['model_state_dict'], strict=True)
split_token_ids = set(checkpoint['run_config']['split_token_ids'])
print('Best epoch:', checkpoint['epoch'])
val_predictions = predict_entities(ner_model, val_loader, encoded_val)
scores, per_label = strict_ner_scores(val_predictions, val_gold)
print('Strict Macro scores:', scores)
for label, row in per_label.items():
    print(label, row)
del checkpoint


In [ ]:
encoded_test = {}
for doc_id, text in test_texts.items():
    ids, offsets = encode_with_offsets(text, tokenizer)
    encoded_test[doc_id] = dict(input_ids=ids, offsets=offsets, labels=[-100]*len(ids))
test_windows = create_windows(encoded_test, MAX_LENGTH, LEFT_CONTEXT)
test_loader = DataLoader(test_windows, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_predictions = predict_entities(ner_model, test_loader, encoded_test)
if set(test_predictions) != set(test_texts):
    raise ValueError('Missing or unexpected test IDs.')
for doc_id, spans in test_predictions.items():
    for start,end,label in spans:
        if not (0 <= start < end <= len(test_texts[doc_id])) or label not in entity_types:
            raise ValueError((doc_id,start,end,label))


In [ ]:
path = PHASE2_OUTPUT_DIR / 'prediction.json'
path.write_text(json.dumps(test_predictions, ensure_ascii=False), encoding='utf-8')
with zipfile.ZipFile(PHASE2_OUTPUT_DIR / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(path, arcname='prediction.json')
print('Saved:', PHASE2_OUTPUT_DIR / 'submission.zip')
